In [53]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

from sklearn.preprocessing import LabelEncoder,label_binarize

import mlflow
import mlflow.sklearn


In [54]:
df = pd.read_csv('D:\Customer Support\Data set\customer_support_tickets_FE.csv')


<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\Windows 10\AppData\Local\Temp\ipykernel_5128\8825122.py:1: SyntaxWarning: invalid escape sequence '\C'
  df = pd.read_csv('D:\Customer Support\Data set\customer_support_tickets_FE.csv')


In [55]:
df['text'] = (
    df['Ticket Subject'].fillna('') +
    ' ' +
    df['Ticket Description'].fillna('')
)


In [56]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    stop_words="english",
    sublinear_tf=True
)
X_text = tfidf.fit_transform(df['text'])

print(X_text.shape)

(8469, 9962)


In [57]:
from scipy.sparse import hstack
from sklearn.preprocessing import LabelEncoder

# Encode categorical features
cat_cols = [
    "Ticket Priority"
]

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    y_type = le.fit_transform(df[col])
    y_priority = le.fit_transform(df[col])

tabular_features = [
    "Customer Age",
    "Customer Gender",
    "Product Purchased",
    "Ticket Channel",
    "Description_Char_Count",
    "Description_Word_Count",
    "Subject_Char_Count",
    "Subject_Word_Count"
]

X_tabular = df[tabular_features].fillna(0)

# Combine TF-IDF + tabular
X_hybrid = hstack([X_text, X_tabular])

In [58]:
X_train_text, X_temp_text, y_train_type, y_temp_type = train_test_split(
    X_text,
    y_type,
    test_size=0.30,
    random_state=42,
    stratify=y_type
)

X_val_text, X_test_text, y_val_type, y_test_type = train_test_split(
    X_temp_text,
    y_temp_type,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_type
)

print(X_train_text.shape)
print(X_val_text.shape)
print(X_test_text.shape)


(5928, 9962)
(1270, 9962)
(1271, 9962)


In [59]:
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('CustomerSupport_Baselines1')


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1781625105172, experiment_id='2', last_update_time=1781625105172, lifecycle_stage='active', name='CustomerSupport_Baselines1', tags={}, trace_location=None, workspace='default'>

In [60]:
with mlflow.start_run(run_name='LogisticRegression_TicketType'):

    lr = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    C=2.0,
    random_state=42
    )

    lr.fit(X_train_text,y_train_type)

    preds = lr.predict(X_test_text)
    probs = lr.predict_proba(X_test_text)

    lr_acc = accuracy_score(y_test_type,preds)

    lr_f1 = f1_score(
        y_test_type,
        preds,
        average='macro'
    )

    y_test_bin = label_binarize(
        y_test_type,
        classes=np.unique(y_type)
    )

    lr_roc = roc_auc_score(
        y_test_bin,
        probs,
        multi_class='ovr'
    )

    mlflow.log_metric('accuracy',lr_acc)
    mlflow.log_metric('f1_macro',lr_f1)
    mlflow.log_metric('roc_auc',lr_roc)

    mlflow.sklearn.log_model(
        lr,
        'logistic_regression'
    )

    print(lr_acc,lr_f1,lr_roc)


2026/06/16 21:33:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 21:33:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.26986624704956724 0.26926316408719486 0.513297363097396
🏃 View run LogisticRegression_TicketType at: http://127.0.0.1:5000/#/experiments/2/runs/d5e41446d7144432985f0115ca970236
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [61]:
with mlflow.start_run(run_name='NaiveBayes_TicketType'):

    nb = MultinomialNB(alpha=0.5)

    nb.fit(X_train_text,y_train_type)

    preds = nb.predict(X_test_text)
    probs = nb.predict_proba(X_test_text)

    nb_acc = accuracy_score(y_test_type,preds)

    nb_f1 = f1_score(
        y_test_type,
        preds,
        average='macro'
    )

    y_test_bin = label_binarize(
        y_test_type,
        classes=np.unique(y_type)
    )

    nb_roc = roc_auc_score(
        y_test_bin,
        probs,
        multi_class='ovr'
    )

    mlflow.log_metric('accuracy',nb_acc)
    mlflow.log_metric('f1_macro',nb_f1)
    mlflow.log_metric('roc_auc',nb_roc)

    mlflow.sklearn.log_model(
        nb,
        'naive_bayes'
    )

    print(nb_acc,nb_f1,nb_roc)


2026/06/16 21:33:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 21:33:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.25177025963808025 0.24595569249986377 0.5117768397341298
🏃 View run NaiveBayes_TicketType at: http://127.0.0.1:5000/#/experiments/2/runs/d4f7b52b7e954ed288fd65b08f944931
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [62]:
for col in [
    'Customer Gender',
    'Product Purchased',
    'Ticket Channel'
]:
    le = LabelEncoder()
    df[col] = le.fit_transform(
        df[col].astype(str)
    )

features = [
    'Customer Age',
    'Customer Gender',
    'Product Purchased',
    'Ticket Channel',
    'Description_Char_Count',
    'Description_Word_Count',
    'Subject_Char_Count',
    'Subject_Word_Count'
]

X_tabular = df[features]

X_train_tab,X_test_tab,y_train_pri,y_test_pri = train_test_split(
    X_tabular,
    y_priority,
    test_size=0.20,
    random_state=42,
    stratify=y_priority
)


In [63]:
with mlflow.start_run(run_name='DecisionTree_TicketPriority'):

    dt = DecisionTreeClassifier(
        criterion="entropy",
        max_depth=20,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    )

    dt.fit(X_train_tab,y_train_pri)

    preds = dt.predict(X_test_tab)
    probs = dt.predict_proba(X_test_tab)

    dt_acc = accuracy_score(y_test_pri,preds)

    dt_f1 = f1_score(
        y_test_pri,
        preds,
        average='macro'
    )

    y_test_bin = label_binarize(
        y_test_pri,
        classes=np.unique(y_priority)
    )

    dt_roc = roc_auc_score(
        y_test_bin,
        probs,
        multi_class='ovr'
    )

    mlflow.log_metric('accuracy',dt_acc)
    mlflow.log_metric('f1_macro',dt_f1)
    mlflow.log_metric('roc_auc',dt_roc)

    mlflow.sklearn.log_model(
        dt,
        'decision_tree'
    )

    print(dt_acc,dt_f1,dt_roc)


2026/06/16 21:33:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 21:33:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.2680047225501771 0.2681471304699278 0.5130540760950066
🏃 View run DecisionTree_TicketPriority at: http://127.0.0.1:5000/#/experiments/2/runs/277509e3e6b544148eaddada8a1d0fc2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [64]:
from sklearn.ensemble import RandomForestClassifier

with mlflow.start_run(run_name="RandomForest_TicketPriority"):

    rf = RandomForestClassifier(
        n_estimators=300,
        max_depth=20,
        min_samples_split=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train_tab, y_train_pri)

    preds = rf.predict(X_test_tab)
    probs = rf.predict_proba(X_test_tab)

    rf_acc = accuracy_score(y_test_pri, preds)

    rf_f1 = f1_score(
        y_test_pri,
        preds,
        average="macro"
    )

    y_test_bin = label_binarize(
        y_test_pri,
        classes=np.unique(y_priority)
    )

    rf_roc = roc_auc_score(
        y_test_bin,
        probs,
        multi_class="ovr"
    )

    mlflow.log_metric("accuracy", rf_acc)
    mlflow.log_metric("f1_macro", rf_f1)
    mlflow.log_metric("roc_auc", rf_roc)

    mlflow.sklearn.log_model(
        rf,
        'Random Forest'
    )

    print(rf_acc, rf_f1, rf_roc)

2026/06/16 21:33:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 21:33:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


0.256198347107438 0.25590374008077793 0.5053298390212777
🏃 View run RandomForest_TicketPriority at: http://127.0.0.1:5000/#/experiments/2/runs/14c14076ae164c679c3d45c8c6c442aa
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [65]:
results = pd.DataFrame({
    'Model':['Logistic Regression','Naive Bayes','Decision Tree',"Random Forest"],
    'Accuracy':[lr_acc,nb_acc,dt_acc,rf_acc],
    'F1_Macro':[lr_f1,nb_f1,dt_f1,rf_f1],
    'ROC_AUC':[lr_roc,nb_roc,dt_roc,rf_roc]
})

results


,Model,Accuracy,F1_Macro,ROC_AUC
0,Logistic Regression,0.269866,0.269263,0.513297
1,Naive Bayes,0.251770,0.245956,0.511777
2,Decision Tree,0.268005,0.268147,0.513054
3,Random Forest,0.256198,0.255904,0.505330
